In [2]:
!pip install -U langchain-community tavily-python

  Using cached langchain_community-0.4.1-py3-none-any.whl.metadata (3.0 kB)
  Using cached langchain_classic-1.0.1-py3-none-any.whl.metadata (4.2 kB)
  Using cached aiohttp-3.13.2-cp311-cp311-win_amd64.whl.metadata (8.4 kB)
  Using cached dataclasses_json-0.6.7-py3-none-any.whl.metadata (25 kB)
  Using cached pydantic_settings-2.12.0-py3-none-any.whl.metadata (3.4 kB)
  Using cached httpx_sse-0.4.3-py3-none-any.whl.metadata (9.7 kB)
  Using cached numpy-2.4.0-cp311-cp311-win_amd64.whl.metadata (6.6 kB)
  Using cached aiohappyeyeballs-2.6.1-py3-none-any.whl.metadata (5.9 kB)
  Using cached aiosignal-1.4.0-py3-none-any.whl.metadata (3.7 kB)
  Using cached frozenlist-1.8.0-cp311-cp311-win_amd64.whl.metadata (21 kB)
  Using cached multidict-6.7.0-cp311-cp311-win_amd64.whl.metadata (5.5 kB)
  Using cached propcache-0.4.1-cp311-cp311-win_amd64.whl.metadata (14 kB)
  Using cached yarl-1.22.0-cp311-cp311-win_amd64.whl.metadata (77 kB)
  Using cached marshmallow-3.26.2-py3-none-any.whl.metadata

In [1]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.tools import Tool  # Correct modern import
from langchain_community.tools.tavily_search import TavilySearchResults


# 🔐 Load API keys from .env
load_dotenv(dotenv_path=".env")

# 🔸 Step 1: Initialize Gemini 2.5 Flash
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash", 
    temperature=0
)

# 🔸 Step 2: Define the QA Tool
qa_prompt = ChatPromptTemplate.from_template("Answer clearly: {question}")
qa_chain = qa_prompt | llm | StrOutputParser()

qa_tool = Tool(
    name="Simple_QA",
    func=qa_chain.invoke, # Use invoke for LCEL
    description="Answer factual questions clearly"
)

# ✅ Tool 3: Web Search (Tavily)
search_tool = Tool(
    name="Web Search",
    func=TavilySearchResults(max_results=3).run,
    description="Search the internet for current information"
)

# 🔸 Step 3: Define the Summarizer Tool
summary_prompt = ChatPromptTemplate.from_template("Summarize the following text briefly:\n\n{text}")
summary_chain = summary_prompt | llm | StrOutputParser()

summary_tool = Tool(
    name="Summarizer",
    func=summary_chain.invoke,
    description="Summarizes long text into a concise version"
)

# 🚀 Step 4: Tool Usage Examples
qa_query = "What is LangGraph in LangChain?"
summary_text = """
LangGraph is a library built on top of LangChain that helps developers create stateful, multi-step agents 
as graphs. Each node represents a step like calling an LLM or a tool. It's ideal for advanced AI workflows.
"""
search_query = "Latest updates on GPT-4o by OpenAI"

# 🖨️ Execution and Output
print("--- TOOL RESULTS ---")

# QA Tool Output
# Note: Since the prompt uses {question}, we pass that key in the dict
print("\n🧠 QA Tool Result:")
print(qa_tool.run({"question": qa_query}))

# Summarizer Tool Output
# Note: Since the prompt uses {text}, we pass that key in the dict
print("\n📝 Summarizer Tool Result:")
print(summary_tool.run({"text": summary_text}))

print("\n🌐 Web Search Tool Output:\n", search_tool.run(search_query))


C:\Users\User\AppData\Local\Temp\ipykernel_12384\3010822401.py:32: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  func=TavilySearchResults(max_results=3).run,


--- TOOL RESULTS ---

🧠 QA Tool Result:
LangGraph is an **extension of the LangChain framework** specifically designed for building **stateful, multi-actor LLM applications with complex control flow.**

Here's a breakdown:

1.  **It's Part of LangChain:** LangGraph is built on top of and seamlessly integrates with LangChain's existing components (LLMs, tools, retrievers, etc.). It uses LangChain primitives as the building blocks for its graph nodes.

2.  **Focus on Graphs, Not Just Chains:**
    *   While LangChain's core `LCEL` (LangChain Expression Language) is excellent for defining linear sequences and simpler conditional logic (chains), **LangGraph takes it a step further by allowing you to define your application as a directed graph.**
    *   This means you can have nodes (steps) and edges (transitions) that represent the flow of execution.

3.  **Key Differentiating Features:**
    *   **Stateful:** LangGraph manages a persistent "state" object that is passed between nodes and 